In [7]:
def avg_equiv_class_size_metric(data, qids, k):
    """
    Return the average sizes of the equivalence classes with respect to the QID set.
    Input:
        data: The input k-anonymized dataframe
        qids: the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return eq_classes.mean() / k


def discernability_metrics(data, qids):
    """
    Return the discernability score of the dataset with respect to the QID set.
    The discernability is calculated by assigning the penalty to each tuple depending 
    on the how many tuples are indistinguishable from it.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return (eq_classes ** 2).sum()


def classification_metrics(data, qids, sensitive_attr):
    """
    Return the classification metric score of the data with respect to the QID set,
    where we assign a penalty to each tuple t. If t's sensitive attribute matches 
    the majority sensitive attribute, the penalty = 0. Otherwise, penalty = size of the equivalence class.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    penalties = 0
    for _, group in data.groupby(list(qids)):
        # class_size = len(group)
        majority = group[sensitive_attr].value_counts().idxmax()
        mismatches = group[group[sensitive_attr] != majority]
        penalties += len(mismatches)
    return penalties / len(data)

In [8]:
import pandas as pd

data = pd.DataFrame({
    "Age": ["20-30", "20-30", "20-30", "30-40", "30-40", "30-40"],
    "Zipcode": ["123**", "123**", "123**", "124**", "124**", "124**"],
    "Disease": ["Flu", "Flu", "Cold", "Cancer", "Cancer", "Flu"]
})

qids = {"Age", "Zipcode"}
k = 3  # k-anonymized dataset (each group size ≥ 3)

# --- Run metrics ---

print("Average Equivalence Class Size Metric:", avg_equiv_class_size_metric(data, qids, k))
print("Discernability Metric:", discernability_metrics(data, qids))
print("Classification Metric:", classification_metrics(data, qids, "Disease"))


Average Equivalence Class Size Metric: 1.0
Discernability Metric: 18
Classification Metric: 0.3333333333333333


In [9]:
# --- Test 1: perfectly homogeneous groups (low penalty expected) ---
data1 = pd.DataFrame({
    "Age": ["20-30"]*3 + ["30-40"]*3,
    "Zipcode": ["123**"]*3 + ["124**"]*3,
    "Disease": ["Flu"]*3 + ["Cancer"]*3
})
print("Test 1: homogeneous groups")
print("CAVG:", avg_equiv_class_size_metric(data1, {"Age","Zipcode"}, k=3))
print("DM:", discernability_metrics(data1, {"Age","Zipcode"}))
print("Classification:", classification_metrics(data1, {"Age","Zipcode"}, "Disease"))
print()

# --- Test 2: mixed sensitive values (moderate penalty) ---
data2 = pd.DataFrame({
    "Age": ["20-30"]*4 + ["30-40"]*4,
    "Zipcode": ["123**"]*4 + ["124**"]*4,
    "Disease": ["Flu","Cold","Flu","Cold", "Cancer","Flu","Cancer","Cold"]
})
print("Test 2: mixed sensitive attributes")
print("CAVG:", avg_equiv_class_size_metric(data2, {"Age","Zipcode"}, k=4))
print("DM:", discernability_metrics(data2, {"Age","Zipcode"}))
print("Classification:", classification_metrics(data2, {"Age","Zipcode"}, "Disease"))
print()

# --- Test 3: uneven group sizes (big + small classes) ---
data3 = pd.DataFrame({
    "Age": ["20-30"]*5 + ["30-40"]*2 + ["40-50"]*3,
    "Zipcode": ["123**"]*5 + ["124**"]*2 + ["125**"]*3,
    "Disease": ["Flu","Cold","Cold","Flu","Cancer","Cancer","Cancer","Flu","Cold","Flu"]
})
print("Test 3: uneven class sizes")
print("CAVG:", avg_equiv_class_size_metric(data3, {"Age","Zipcode"}, k=2))
print("DM:", discernability_metrics(data3, {"Age","Zipcode"}))
print("Classification:", classification_metrics(data3, {"Age","Zipcode"}, "Disease"))
print()

# --- Test 4: large messy dataset (randomized) ---
import numpy as np
np.random.seed(0)
ages = np.random.choice(["20-30","30-40","40-50"], 30)
zips = np.random.choice(["123**","124**","125**"], 30)
diseases = np.random.choice(["Flu","Cold","Cancer"], 30)
data4 = pd.DataFrame({"Age":ages,"Zipcode":zips,"Disease":diseases})

print("Test 4: randomized 30 records")
print("CAVG:", avg_equiv_class_size_metric(data4, {"Age","Zipcode"}, k=2))
print("DM:", discernability_metrics(data4, {"Age","Zipcode"}))
print("Classification:", classification_metrics(data4, {"Age","Zipcode"}, "Disease"))


Test 1: homogeneous groups
CAVG: 1.0
DM: 18
Classification: 0.0

Test 2: mixed sensitive attributes
CAVG: 1.0
DM: 32
Classification: 0.5

Test 3: uneven class sizes
CAVG: 1.6666666666666667
DM: 38


Classification: 0.4

Test 4: randomized 30 records
CAVG: 1.6666666666666667
DM: 132
Classification: 0.3


In [ ]:
import pandas as pd
import numpy as np
import unittest
from unittest.mock import patch

# Original functions (assuming they're imported or defined here)
def avg_equiv_class_size_metric(data, qids, k):
    """
    Return the average sizes of the equivalence classes with respect to the QID set.
    Input:
        data: The input k-anonymized dataframe
        qids: the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return eq_classes.mean() / k

def discernability_metrics(data, qids):
    """
    Return the discernability score of the dataset with respect to the QID set.
    The discernability is calculated by assigning the penalty to each tuple depending
    on the how many tuples are indistinguishable from it.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return (eq_classes ** 2).sum()

def classification_metrics(data, qids, sensitive_attr):
    """
    Return the classification metric score of the data with respect to the QID set,
    where we assign a penalty to each tuple t. If t's sensitive attribute matches
    the majority sensitive attribute, the penalty = 0. Otherwise, penalty = size of the equivalence class.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    penalties = 0
    for _, group in data.groupby(list(qids)):
        # class_size = len(group)
        majority = group[sensitive_attr].value_counts().idxmax()
        mismatches = group[group[sensitive_attr] != majority]
        penalties += len(mismatches)
    return penalties / len(data)


class TestPrivacyMetrics(unittest.TestCase):
    
    def setUp(self):
        """Set up test data for various scenarios"""
        # Basic test data
        self.basic_data = pd.DataFrame({
            'age': [25, 25, 30, 30, 35, 35],
            'zip': ['12345', '12345', '67890', '67890', '11111', '11111'],
            'salary': ['high', 'low', 'high', 'high', 'low', 'low']
        })
        
        # Perfect k=2 anonymization
        self.perfect_k2_data = pd.DataFrame({
            'age': [25, 25, 30, 30, 35, 35, 40, 40],
            'zip': ['123*', '123*', '678*', '678*', '111*', '111*', '999*', '999*'],
            'disease': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B']
        })
        
        # Uneven equivalence classes
        self.uneven_data = pd.DataFrame({
            'age': [25, 25, 25, 30, 30, 35],
            'city': ['NYC', 'NYC', 'NYC', 'LA', 'LA', 'SF'],
            'condition': ['X', 'Y', 'X', 'X', 'Y', 'X']
        })
        
        # Single equivalence class (worst case for discernability)
        self.single_class_data = pd.DataFrame({
            'age': [30, 30, 30, 30],
            'zip': ['*****', '*****', '*****', '*****'],
            'status': ['healthy', 'sick', 'healthy', 'healthy']
        })
        
        # Each record is unique (worst case for k-anonymity)
        self.unique_data = pd.DataFrame({
            'age': [25, 26, 27, 28],
            'zip': ['12345', '12346', '12347', '12348'],
            'disease': ['A', 'B', 'A', 'B']
        })
        
        # Edge case: empty dataframe
        self.empty_data = pd.DataFrame()
        
        # Edge case: single row
        self.single_row_data = pd.DataFrame({
            'age': [25],
            'zip': ['12345'],
            'status': ['healthy']
        })

    def test_avg_equiv_class_size_metric_basic(self):
        """Test basic functionality of average equivalence class size"""
        result = avg_equiv_class_size_metric(
            self.basic_data, {'age', 'zip'}, k=2
        )
        # 3 classes, each with 2 records: avg = 2, normalized by k=2 -> 1.0
        self.assertEqual(result, 1.0)
        
    def test_avg_equiv_class_size_metric_different_k(self):
        """Test with different k values"""
        result_k1 = avg_equiv_class_size_metric(
            self.basic_data, {'age', 'zip'}, k=1
        )
        result_k3 = avg_equiv_class_size_metric(
            self.basic_data, {'age', 'zip'}, k=3
        )
        # Same avg class size (2), different normalization
        self.assertAlmostEqual(result_k1, 2.0)
        self.assertAlmostEqual(result_k3, 2.0/3)
        
    def test_avg_equiv_class_size_metric_uneven(self):
        """Test with uneven equivalence classes"""
        result = avg_equiv_class_size_metric(
            self.uneven_data, {'age', 'city'}, k=2
        )
        # Classes: size 3, 2, 1 -> avg = 2, normalized by k=2 -> 1.0
        expected = (3 + 2 + 1) / 3 / 2  # avg class size / k
        self.assertAlmostEqual(result, expected)
        
    def test_avg_equiv_class_size_metric_unique_records(self):
        """Test when all records are unique"""
        result = avg_equiv_class_size_metric(
            self.unique_data, {'age', 'zip'}, k=2
        )
        # 4 classes, each with 1 record: avg = 1, normalized by k=2 -> 0.5
        self.assertEqual(result, 0.5)

    def test_discernability_metrics_basic(self):
        """Test basic discernability calculation"""
        result = discernability_metrics(self.basic_data, {'age', 'zip'})
        # 3 classes, each with 2 records: 2² + 2² + 2² = 12
        self.assertEqual(result, 12)
        
    def test_discernability_metrics_uneven(self):
        """Test discernability with uneven classes"""
        result = discernability_metrics(self.uneven_data, {'age', 'city'})
        # Classes: size 3, 2, 1 -> 3² + 2² + 1² = 9 + 4 + 1 = 14
        self.assertEqual(result, 14)
        
    def test_discernability_metrics_single_class(self):
        """Test worst case: single equivalence class"""
        result = discernability_metrics(self.single_class_data, {'age', 'zip'})
        # 1 class with 4 records: 4² = 16
        self.assertEqual(result, 16)
        
    def test_discernability_metrics_unique(self):
        """Test best case: all records unique"""
        result = discernability_metrics(self.unique_data, {'age', 'zip'})
        # 4 classes, each with 1 record: 1² + 1² + 1² + 1² = 4
        self.assertEqual(result, 4)

    def test_classification_metrics_basic(self):
        """Test basic classification metric"""
        result = classification_metrics(self.basic_data, {'age', 'zip'}, 'salary')
        # Class 1 (age=25, zip=12345): ['high', 'low'] -> majority='high' (tie), 1 mismatch
        # Class 2 (age=30, zip=67890): ['high', 'high'] -> majority='high', 0 mismatches  
        # Class 3 (age=35, zip=11111): ['low', 'low'] -> majority='low', 0 mismatches
        # Total penalties: 1, normalized by total records (6) = 1/6
        expected = 1/6
        self.assertAlmostEqual(result, expected, places=6)
        
    def test_classification_metrics_perfect_diversity(self):
        """Test with perfect diversity in sensitive attribute"""
        result = classification_metrics(
            self.perfect_k2_data, {'age', 'zip'}, 'disease'
        )
        # Each class has equal distribution of A and B
        # 4 classes, each with 2 records (1 A, 1 B), so 1 mismatch per class
        # Total penalties: 4, normalized by total records (8) = 4/8 = 0.5
        self.assertEqual(result, 0.5)
        
    def test_classification_metrics_no_diversity(self):
        """Test when all records in each class have same sensitive value"""
        uniform_data = pd.DataFrame({
            'age': [25, 25, 30, 30],
            'zip': ['123*', '123*', '678*', '678*'],
            'disease': ['A', 'A', 'B', 'B']
        })
        result = classification_metrics(uniform_data, {'age', 'zip'}, 'disease')
        # Each class is homogeneous, so no penalties
        self.assertEqual(result, 0.0)

    def test_edge_cases_single_row(self):
        """Test edge case with single row"""
        # avg_equiv_class_size_metric
        result1 = avg_equiv_class_size_metric(
            self.single_row_data, {'age', 'zip'}, k=1
        )
        self.assertEqual(result1, 1.0)
        
        # discernability_metrics  
        result2 = discernability_metrics(self.single_row_data, {'age', 'zip'})
        self.assertEqual(result2, 1)
        
        # classification_metrics
        result3 = classification_metrics(
            self.single_row_data, {'age', 'zip'}, 'status'
        )
        self.assertEqual(result3, 0.0)  # Single record is always majority

    def test_empty_qids_set(self):
        """Test behavior with empty QIDs set"""
        # This should group all records into one equivalence class
        result1 = avg_equiv_class_size_metric(self.basic_data, set(), k=2)
        expected1 = len(self.basic_data) / 2  # all records in one class
        self.assertEqual(result1, expected1)
        
        result2 = discernability_metrics(self.basic_data, set())
        expected2 = len(self.basic_data) ** 2  # all records in one class
        self.assertEqual(result2, expected2)

    def test_nonexistent_sensitive_attribute(self):
        """Test classification metric with non-existent sensitive attribute"""
        with self.assertRaises(KeyError):
            classification_metrics(
                self.basic_data, {'age', 'zip'}, 'nonexistent_column'
            )

    def test_nonexistent_qid_columns(self):
        """Test with non-existent QID columns"""
        with self.assertRaises(KeyError):
            avg_equiv_class_size_metric(
                self.basic_data, {'nonexistent_col'}, k=2
            )
        
        with self.assertRaises(KeyError):
            discernability_metrics(self.basic_data, {'nonexistent_col'})

    def test_negative_k_value(self):
        """Test behavior with negative k"""
        result = avg_equiv_class_size_metric(
            self.basic_data, {'age', 'zip'}, k=-2
        )
        # Should still compute but result will be negative
        self.assertLess(result, 0)

    def test_data_types_and_mixed_types(self):
        """Test with various data types"""
        mixed_data = pd.DataFrame({
            'int_col': [1, 1, 2, 2],
            'float_col': [1.5, 1.5, 2.5, 2.5], 
            'str_col': ['a', 'b', 'a', 'b'],
            'bool_col': [True, True, False, False]
        })
        
        # Test with mixed QID types
        result1 = avg_equiv_class_size_metric(
            mixed_data, {'int_col', 'float_col'}, k=2
        )
        self.assertEqual(result1, 1.0)
        
        result2 = discernability_metrics(mixed_data, {'int_col', 'float_col'})
        self.assertEqual(result2, 8)  # 2 classes, each size 2: 2² + 2² = 8
        
        result3 = classification_metrics(
            mixed_data, {'int_col', 'float_col'}, 'str_col'
        )
        self.assertEqual(result3, 0.5)  # Each class has 1 mismatch

    def test_large_dataset_performance(self):
        """Test with larger dataset for performance"""
        # Create larger dataset
        np.random.seed(42)
        large_data = pd.DataFrame({
            'age': np.random.randint(20, 70, 1000),
            'zip': np.random.randint(10000, 99999, 1000),
            'income': np.random.choice(['low', 'medium', 'high'], 1000)
        })
        
        # These should complete without issues
        result1 = avg_equiv_class_size_metric(large_data, {'age'}, k=5)
        result2 = discernability_metrics(large_data, {'age'})
        result3 = classification_metrics(large_data, {'age'}, 'income')
        
        # Basic sanity checks
        self.assertGreater(result1, 0)
        self.assertGreater(result2, 0) 
        self.assertGreaterEqual(result3, 0)
        self.assertLessEqual(result3, 1)

    def test_string_vs_set_qids(self):
        """Test behavior with QIDs as list vs set"""
        # Should work the same way
        result_set = avg_equiv_class_size_metric(
            self.basic_data, {'age', 'zip'}, k=2
        )
        result_list = avg_equiv_class_size_metric(
            self.basic_data, ['age', 'zip'], k=2
        )
        self.assertEqual(result_set, result_list)


def run_comprehensive_tests():
    """Run all tests and provide summary"""
    # Create test suite
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestPrivacyMetrics)
    
    # Run tests
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    
    # Print summary
    print(f"\n{'='*50}")
    print(f"TEST SUMMARY")
    print(f"{'='*50}")
    print(f"Tests run: {result.testsRun}")
    print(f"Failures: {len(result.failures)}")
    print(f"Errors: {len(result.errors)}")
    print(f"Success rate: {(result.testsRun - len(result.failures) - len(result.errors))/result.testsRun*100:.1f}%")
    
    if result.failures:
        print(f"\nFAILURES:")
        for test, traceback in result.failures:
            print(f"- {test}: {traceback.split('AssertionError: ')[-1].split('\\n')[0]}")
    
    if result.errors:
        print(f"\nERRORS:")
        for test, traceback in result.errors:
            print(f"- {test}: {traceback.split('\\n')[-2]}")
    
    return result.wasSuccessful()


if __name__ == "__main__":
    # Run the comprehensive test suite
    success = run_comprehensive_tests()
    
    # Additional manual verification examples
    print(f"\n{'='*50}")
    print(f"MANUAL VERIFICATION EXAMPLES")
    print(f"{'='*50}")
    
    # Example 1: Perfect k=2 anonymization
    perfect_data = pd.DataFrame({
        'age': [25, 25, 30, 30],
        'city': ['NYC', 'NYC', 'LA', 'LA'],
        'disease': ['A', 'B', 'A', 'B']
    })
    
    print("Example 1: Perfect k=2 anonymization")
    print("Data:")
    print(perfect_data)
    print(f"Avg equiv class size (k=2): {avg_equiv_class_size_metric(perfect_data, {'age', 'city'}, 2)}")
    print(f"Discernability: {discernability_metrics(perfect_data, {'age', 'city'})}")
    print(f"Classification metric: {classification_metrics(perfect_data, {'age', 'city'}, 'disease')}")
    
    # Example 2: Poor anonymization
    poor_data = pd.DataFrame({
        'age': [25, 26, 27, 28],
        'city': ['NYC', 'NYC', 'LA', 'LA'],
        'disease': ['A', 'A', 'B', 'B']
    })
    
    print(f"\nExample 2: Poor anonymization (unique ages)")
    print("Data:")
    print(poor_data)
    print(f"Avg equiv class size (k=2): {avg_equiv_class_size_metric(poor_data, {'age', 'city'}, 2)}")
    print(f"Discernability: {discernability_metrics(poor_data, {'age', 'city'})}")
    print(f"Classification metric: {classification_metrics(poor_data, {'age', 'city'}, 'disease')}")
    
    print(f"\n{'='*50}")
    print("Testing completed successfully!" if success else "Some tests failed - see details above")
    print(f"{'='*50}")

test_avg_equiv_class_size_metric_basic (__main__.TestPrivacyMetrics.test_avg_equiv_class_size_metric_basic)
Test basic functionality of average equivalence class size ... 

ok
test_avg_equiv_class_size_metric_different_k (__main__.TestPrivacyMetrics.test_avg_equiv_class_size_metric_different_k)
Test with different k values ... ok
test_avg_equiv_class_size_metric_uneven (__main__.TestPrivacyMetrics.test_avg_equiv_class_size_metric_uneven)
Test with uneven equivalence classes ... ok
test_avg_equiv_class_size_metric_unique_records (__main__.TestPrivacyMetrics.test_avg_equiv_class_size_metric_unique_records)
Test when all records are unique ... ok
test_classification_metrics_basic (__main__.TestPrivacyMetrics.test_classification_metrics_basic)
Test basic classification metric ... ok
test_classification_metrics_no_diversity (__main__.TestPrivacyMetrics.test_classification_metrics_no_diversity)
Test when all records in each class have same sensitive value ... ok
test_classification_metrics_perfect_diversity (__main__.TestPrivacyMetrics.test_classification_metrics_perfect_diversity)
Test with perfect diversity in sensitive attribute ... ok
test_data_types_and_mi


TEST SUMMARY
Tests run: 20
Failures: 1
Errors: 1
Success rate: 90.0%

FAILURES:
- test_zero_k_value (__main__.TestPrivacyMetrics.test_zero_k_value): ZeroDivisionError not raised


ERRORS:


IndexError: list index out of range